# Step 1 — what data is actually in Drive?

**Just press Run on each cell in order.** Nothing to edit, no paths to type.

Cell 1 mounts Drive and finds every folder holding data files by itself.
Cell 2 shows the columns of one L2 file — that decides how the rest of the work gets written.

Nothing here downloads or copies anything. It only reads.

In [ ]:
# CELL 1 — mount Drive, then auto-find every folder that holds data files.
# A popup will ask you to authorise Drive access: click through it.

from google.colab import drive
drive.mount('/content/drive')

import os, re
from collections import defaultdict

DATA_EXT = ('.csv', '.parquet', '.gz', '.zip', '.pkl', '.db', '.json')
ROOT = '/content/drive/MyDrive'

found = []
for cur, _dirs, files in os.walk(ROOT):
    hits = [f for f in files if f.lower().endswith(DATA_EXT)]
    if len(hits) < 3:          # skip folders with a stray file or two
        continue
    try:
        total = sum(os.path.getsize(os.path.join(cur, f)) for f in hits)
    except OSError:
        continue
    # pull YYYYMMDD or YYYY-MM-DD out of filenames to get a date range
    dates = sorted(m.group(0).replace('-', '')
                   for f in hits
                   if (m := re.search(r'\d{4}-?\d{2}-?\d{2}', f)))
    found.append((total, cur, len(hits), dates))

found.sort(reverse=True)          # biggest first
print(f'{len(found)} data folders found\n' + '=' * 78)
for total, path, n, dates in found[:40]:
    rel = os.path.relpath(path, ROOT)
    span = f'{dates[0]} -> {dates[-1]}' if dates else 'no dates in filenames'
    print(f'{total/1e9:8.2f} GB  {n:>6} files  {span}')
    print(f'           {rel}')

print('\n' + '=' * 78)
print('Copy the folder line you think is the NQ L2 data — Cell 2 uses the biggest by default.')

In [ ]:
# CELL 2 — what IS an L2 file? Everything downstream depends on these column names.
#
# By default this probes the LARGEST folder from Cell 1. If that isn't the L2 data,
# set PICK to the number of the folder you want (0 = first line printed, 1 = second, ...).

PICK = 0

import pandas as pd

_total, folder, _n, _dates = found[PICK]
files = sorted(os.path.join(folder, f) for f in os.listdir(folder)
               if f.lower().endswith(DATA_EXT))
print('folder :', folder)
print('files  :', len(files))
print('first  :', os.path.basename(files[0]))
print('last   :', os.path.basename(files[-1]))

f = files[0]
head = pd.read_parquet(f).head(3) if f.endswith('.parquet') else pd.read_csv(f, nrows=3)
print('\ncolumns:', head.columns.tolist())
print()
print(head.T.to_string())

# rows in the whole file -> tells us the snapshot frequency
if not f.endswith('.parquet'):
    with open(f) as fh:
        nrows = sum(1 for _ in fh) - 1
    print(f'\nrows in this file: {nrows:,}')
    print(f'~{nrows/23400:.1f} rows/sec if it covers a 6.5h session')

## Paste both outputs back to Claude

What matters:

1. **Cell 1** — the full folder list. Confirms how many L2 days exist and their date range,
   and finds the other datasets (`oi_qqq`, `second_order_qqq`, L1, gold L2) at the same time.
2. **Cell 2** — the column names. This is the fork in the road:
   - `bid_px_00..09` / `bid_sz_00..09` → depth snapshots, order-book imbalance is a direct
     column sum. Easy.
   - `order_id` / `action` → per-order events, the book has to be replayed. Much harder.

   You said there's no `order_id`, so the easy path is expected — this confirms it and gives
   the exact spelling of the columns.